In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(42)

In [3]:
# num_experts
# top_k
# d_model
# dropout

In [2]:
class Expert(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.ffwd = nn.Sequential(
        nn.Linear(config.d_model, 4*config.d_model),
        nn.ReLU(),
        nn.Linear(4*config.d_model, config.d_model),
        nn.Dropout(config.dropout)
    )

  def forward(self, x):
    return self.ffwd(x)

In [5]:
class TopKRouter(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.top_k = config.top_k
    self.linear = nn.Linear(config.d_model, config.num_experts) # routing matrix
    self.noise_linear = nn.Linear(config.d_model, config.num_experts)

  def forward(self, x):
    # x is the output tensor from the mha block (x: (B, T, C))
    logits = self.linear(x) # expert selecter matric
    noise_logits = self.noise_linear(x)

    # softplus is a smooth version of ReLU and is always positive
    noise = torch.randn_like(logits) * F.softplus(noise_logits) # (B, T, num_experts)
    noisy_logits = logits + noise

    top_k_logits, indices = logits.top_k(self.top_k, dim=-1)
    zeros_matrix = torch.full_like(top_k_logits, float("-inf"))
    sparse_logits = zeros_matrix.scatter(dim=-1, index=indices, src=top_k_logits)
    router_output = F.softmax(sparse_logits, dim=-1)
    return router_output, indices

In [ ]:
class SparseMoE(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.router = NoisyTopKRouter(config)
    self.experts = nn.ModuleList([Expert(config) for _ in range(config.num_experts)])
    self.top_k = config.top_k

  def forward(self, x):
    # x: (B, T, C)
    gating_output, indices = self.router(x) # (B, T, num_experts), (B, T, num_experts)
    final_output = torch.zeros_like(x) # (B, T, C)

    flat_x = x.view(-1, x.size(-1)) # (B*T, d_model)
    # gating_output contains the routing weights for every token
    flat_gating_output = gating_output.view(-1, gating_output.size(-1)) # (B*T, num_experts)

    # process each expert in parallel
    for i, expert in self.enumerate(self.experts):
      # indices (B, T, top_k)
      expert_mask = (indices == i).any(dim=-1) # boolean mask (B, T)
      flat_mask = expert_mask.view(-1) # (B*T)

      if flat_mask.any():
        expert_input = flat_x[flat_mask] # (B*T, d_model) keeps token embeddings where flat_mask == True
        expert_output = expert(expert_input) # (B*T, d_model)

        # extract and apply gating scores 
        gating_scores = flat_gating_output[flat_mask, i].unsqueeze(1) # (B*T, 1) keeps rows where flat_mask == True
        weighted_output = expert_output * gating_scores # (B*T, d_model) 
        final_output[expert_mask] += weighted_output.squeeze(1) # selected tokens (B, T, d_model)

    return final_output

In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.d_model = config.d_model
    self.n_heads = config.n_heads
    self.head_size = self.d_model // self.n_heads
    self.W_q = nn.Linear(self.d_model, self.d_model)
    self.W_k = nn.Linear(self.d_model, self.d_model)
    self.W_v = nn.Linear(self.d_model, self.d_model)
    self.W_o = nn.Linear(self.d_model, self.d_model)

  def scaled_dot_product(self, Q, K, V, mask=None):
    # Q, K, V: (B, n_heads, T, head_size)
    attn_scores = Q @ K.transpose(-1, -2) # (B, n_heads, T, head_size) @ (B, n_heads, head_size, T) => (B, n_heads, T, T)
    attn_scores = attn_scores / math.sqrt(self.head_size)
    if mask is not None:
      attn_scores = attn_scores.masked_fill(mask==0, float("-inf"))
    attn_probs = torch.softmax(attn_scores, dim=-1)
    output = attn_probs @ V # (B, n_heads, T, T) @ (B, n_heads, T, head_size) => (B, n_heads, T, head_size)
    return output

  def split_heads(self, x):
    B, T, C = x.size() # (B, T, d_model)
    return x.view(B, T, self.n_heads, self.head_size).transpose(1, 2) # (B, n_heads, T, head_size)

  def combine_heads(self, x):
    B, n_heads, T, head_size = x.size() # (B, n_heads, T, head_size)
    return x.transpose(1, 2).view(B, T, n_heads*head_size) # (B, T, head_size)

  def forward(self, Q, K, V, mask=None):
    Q = self.split_heads(self.W_q(Q)) # (B, T, d_model) @ (d_model, d_model) => (B, T, d_model) => (B, n_heads, T, head_size)
    K = self.split_heads(self.W_k(K)) # (B, T, d_model) @ (d_model, d_model) => (B, T, d_model) => (B, n_heads, T, head_size)
    V = self.split_heads(self.W_v(V)) # (B, T, d_model) @ (d_model, d_model) => (B, T, d_model) => (B, n_heads, T, head_size)
    attn_output = self.scaled_dot_product(Q, K, V, mask)
    output = self.W_o(self.combine_heads(attn_output)) # (B, T, d_model) @ (d_model, d_model) => (B, T, d_model)
    return output